# Clean and Normalize Product Dataset

Notebook này chỉ đọc dữ liệu trong `data/`, làm sạch, chuẩn hóa và xuất kết quả vào folder `processed/`.

Không tạo dữ liệu train, không train model.

In [1]:
import json
import html
import re
import unicodedata
from pathlib import Path

import pandas as pd

DATA_DIR = Path("/Users/huybui/Desktop/Smart Shopping Chatbot/data")
OUT_DIR = Path("/Users/huybui/Desktop/Smart Shopping Chatbot/processed")
OUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)

## 1. Read All CSV Files

In [2]:
csv_files = sorted(DATA_DIR.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files")
for file in csv_files:
    print(file)

Found 10 CSV files
/Users/huybui/Desktop/Smart Shopping Chatbot/data/thong_tin_links_accessory.csv
/Users/huybui/Desktop/Smart Shopping Chatbot/data/thong_tin_links_bluetooth.csv
/Users/huybui/Desktop/Smart Shopping Chatbot/data/thong_tin_links_laptop.csv
/Users/huybui/Desktop/Smart Shopping Chatbot/data/thong_tin_links_micro.csv
/Users/huybui/Desktop/Smart Shopping Chatbot/data/thong_tin_links_pc.csv
/Users/huybui/Desktop/Smart Shopping Chatbot/data/thong_tin_links_san_pham.csv
/Users/huybui/Desktop/Smart Shopping Chatbot/data/thong_tin_links_screen.csv
/Users/huybui/Desktop/Smart Shopping Chatbot/data/thong_tin_links_speaker.csv
/Users/huybui/Desktop/Smart Shopping Chatbot/data/thong_tin_links_tablet.csv
/Users/huybui/Desktop/Smart Shopping Chatbot/data/thong_tin_products_links.csv


In [3]:
def category_from_filename(path: Path) -> str:
    mapping = {
        "thong_tin_links_san_pham": "dien_thoai",
        "thong_tin_products_links": "dien_thoai",
        "thong_tin_links_laptop": "laptop",
        "thong_tin_links_pc": "pc",
        "thong_tin_links_screen": "man_hinh",
        "thong_tin_links_speaker": "loa",
        "thong_tin_links_tablet": "may_tinh_bang",
        "thong_tin_links_bluetooth": "tai_nghe_bluetooth",
        "thong_tin_links_micro": "micro",
        "thong_tin_links_accessory": "phu_kien",
    }
    return mapping.get(path.stem.lower(), path.stem.lower())

frames = []
for file in csv_files:
    df = pd.read_csv(file, encoding="utf-8-sig", dtype=str, keep_default_na=False)
    df["source_file"] = file.name
    df["category"] = category_from_filename(file)
    frames.append(df)

raw = pd.concat(frames, ignore_index=True, sort=False).fillna("")
print(raw.shape)
raw.head()

(3037, 139)


,url,ten_san_pham,thuong_hieu,gia_ban,sku,mo_ta_ngan,Loại case,Hỗ trợ quạt,Số khe cắm PCI,Kích thước,Hãng sản xuất,Loại tản nhiệt,Số vòng quay của quạt,Độ ồn (dBA),Kết nối,Chipset,Socket,Khe RAM tối đa,Kiểu RAM hỗ trợ,Dung lượng RAM tối đa,Số cổng USB,LAN,Âm thanh,Loại ổ cứng,Kết nối,Dung lượng,Tốc độ đọc,Hỗ trợ hệ điều hành,Công suất nguồn,Quạt làm mát,Tiêu chuẩn nguồn,Dung lượng ổ cứng,Chất liệu,Tính năng khác,Loại RAM,Bus RAM,Hỗ trợ,Độ trễ,Điện áp,Đèn LED,...,Phạm vi sử dụng,Tỷ lệ lấy mẫu,Tỷ lệ tín hiệu trên tạp âm,Tần số đáp ứng,Dòng camera,Lõi pin,CPU,VGA,RAM,Cổng I/O mặt sau,SSD,Mainboard,Chipset (PC lắp ráp),Màn hình,Camera sau,Camera trước,Công nghệ NFC,Bộ nhớ trong,Thẻ SIM,Tính năng màn hình,Cảm biến,Tương thích,Kích thước thực tế (bao gồm viền),Tấm nền,Tỉ lệ màn hình,Tần số quét,Thời gian phản hồi,Cổng kết nối,Trọng lượng,Độ tương phản động,Chuẩn VESA,Góc xoay dọc,Góc xoay ngang,Kích cỡ màn hình hỗ trợ,Tải trọng màn hình tối đa,Góc lật màn hình,Độ dày bàn phù hợp,Chống nước,Số loa kết nối cùng lúc,Cấu tạo
0,https://cellphones.com.vn/case-may-tinh-msi-mag-forge-320r-airflow.html,Case máy tính MSI MAG FORGE 320R AIRFLOW,MSI,Liên hệ,CMT.MSI.02,"Mua Case máy tính MSI MAG FORGE 320R AIRFLOW chính hãng - Giá rẻ, đảm bảo chất lượng, độ bền cao, hỗ trợ giao hàng tận nơi toàn quốc.",Mid Tower,Mặt trước: 3 x 120 mm / 2 x 140 mmMặt trên: 3 x 120 mm / 2 x 140 mmMặt sau: 1 x 120 mmMặt bên: 2 x 120 mmVỏ bọc PSU: 2 x 120 mm,7,472.5 x 210 x 498 mm,MSI,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,...,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,https://cellphones.com.vn/tan-nhiet-khi-cpu-jonsbo-cr-1200e-rgb.html,Tản nhiệt khí CPU Jonsbo CR-1200E RGB,Jonsbo,Liên hệ,tan-nhiet-khi-cpu-jonsbo-cr-1200e-rgb,"Mua Tản nhiệt khí CPU Jonsbo CR-1200E RGB chính hãng - Giá rẻ, đảm bảo chất lượng, độ bền cao, hỗ trợ giao hàng tận nơi toàn quốc.",,,,,Jonsbo,Tản nhiệt khí,2000 RPM ±10%,≤ 26.5 dB(A),1 x 3-pin,,,,,,,,,,,,,,,,,,,,,,,,,,...,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2,https://cellphones.com.vn/mainboard-msi-b450m-a-pro-max-ii.html,Mainboard MSI B450M-A PRO MAX II,MSI,Liên hệ,MB.MSI.12,"Mua Mainboard MSI B450M-A PRO MAX II chính hãng - Giá rẻ, đảm bảo chất lượng, độ bền cao, hỗ trợ giao hàng tận nơi toàn quốc.",,,,M-ATX,MSI,,,,,AMD,AM4,2,DDR4,64GB,2x USB 2.0 (Phía sau) 4x USB 2.0 (Phía trước) 4x USB 5Gbps Type A (Phía sau) 2x USB 5Gbps Type A (Phía trước),Bộ điều khiển LAN Realtek 8125 2.5Gbps,Realtek ALC897 Codec Âm thanh độ nét cao 7.1 kênh,,,,,,,,,,,,,,,,,,...,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3,https://cellphones.com.vn/o-cung-ssd-transcend-110s-nvme-pcie-gen3-x4-512gb.html,Ổ cứng SSD Transcend 110S NVMe PCIe Gen3 x4 512GB,Transcend,Liên hệ,OC.TC.42,"Ổ cứng SSD Transcend 110S NVMe PCIe Gen3 x4 512GB chính hãng - Giá rẻ, đảm bảo chất lượng, độ bền cao, hỗ trợ giao hàng tận nơi toàn quốc.",,,,,Transcend,,,,,,,,,,,,,SSD,M2 PCIe,512GB,1700 MB/s,Microsoft Windows 7 trở lênLinux Kernel 2.6.31 trở lên,,,,,,,,,,,,,...,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
4,https://cellphones.com.vn/nguon-may-tinh-msi-mag-a850gl-pcie5-850w.html,Nguồn máy tính MSI Mag A850GL PCIe5 850W (80 Plus Gold),MSI,Liên hệ,nguon-may-tinh-msi-mag-a850gl-pcie5-850w,"Mua nguồn máy tính MSI Mag A850GL PCIe5 850W (80 Plus Gold) chính hãng - Giá rẻ, đảm bảo chất lượng, độ bền cao, hỗ trợ giao hàng tận nơi toàn quốc.",,,,,MSI,,,,,,,,,,,,,,,,,,850W,1 x Fan 12 cm,80 Plus Gold,,,,,,,,,,...,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,


## 2. Clean Text and Column Names

In [4]:
def clean_text(value) -> str:
    if pd.isna(value):
        return ""
    text = str(value).replace("\ufeff", "")
    text = html.unescape(text)
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\\[rnt]", " ", text)
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip()

def normalize_column_name(name: str) -> str:
    text = clean_text(name).lower()
    text = re.sub(r"\s+", "_", text)
    text = re.sub(r"_+", "_", text)
    return text.strip("_")

def make_unique_columns(columns: list) -> list:
    seen = {}
    unique = []
    for col in columns:
        count = seen.get(col, 0) + 1
        seen[col] = count
        unique.append(col if count == 1 else f"{col}_{count}")
    return unique

clean = raw.copy()
clean.columns = make_unique_columns([normalize_column_name(col) for col in clean.columns])

for col in clean.columns:
    clean[col] = clean[col].map(clean_text)

clean.head()

,url,ten_san_pham,thuong_hieu,gia_ban,sku,mo_ta_ngan,loại_case,hỗ_trợ_quạt,số_khe_cắm_pci,kích_thước,hãng_sản_xuất,loại_tản_nhiệt,số_vòng_quay_của_quạt,độ_ồn_(dba),kết_nối,chipset,socket,khe_ram_tối_đa,kiểu_ram_hỗ_trợ,dung_lượng_ram_tối_đa,số_cổng_usb,lan,âm_thanh,loại_ổ_cứng,kết_nối_2,dung_lượng,tốc_độ_đọc,hỗ_trợ_hệ_điều_hành,công_suất_nguồn,quạt_làm_mát,tiêu_chuẩn_nguồn,dung_lượng_ổ_cứng,chất_liệu,tính_năng_khác,loại_ram,bus_ram,hỗ_trợ,độ_trễ,điện_áp,đèn_led,...,phạm_vi_sử_dụng,tỷ_lệ_lấy_mẫu,tỷ_lệ_tín_hiệu_trên_tạp_âm,tần_số_đáp_ứng,dòng_camera,lõi_pin,cpu,vga,ram,cổng_i/o_mặt_sau,ssd,mainboard,chipset_(pc_lắp_ráp),màn_hình,camera_sau,camera_trước,công_nghệ_nfc,bộ_nhớ_trong,thẻ_sim,tính_năng_màn_hình,cảm_biến,tương_thích,kích_thước_thực_tế_(bao_gồm_viền),tấm_nền,tỉ_lệ_màn_hình,tần_số_quét,thời_gian_phản_hồi,cổng_kết_nối_2,trọng_lượng_2,độ_tương_phản_động,chuẩn_vesa,góc_xoay_dọc,góc_xoay_ngang,kích_cỡ_màn_hình_hỗ_trợ,tải_trọng_màn_hình_tối_đa,góc_lật_màn_hình,độ_dày_bàn_phù_hợp,chống_nước,số_loa_kết_nối_cùng_lúc,cấu_tạo
0,https://cellphones.com.vn/case-may-tinh-msi-mag-forge-320r-airflow.html,Case máy tính MSI MAG FORGE 320R AIRFLOW,MSI,Liên hệ,CMT.MSI.02,"Mua Case máy tính MSI MAG FORGE 320R AIRFLOW chính hãng - Giá rẻ, đảm bảo chất lượng, độ bền cao, hỗ trợ giao hàng tận nơi toàn quốc.",Mid Tower,Mặt trước: 3 x 120 mm / 2 x 140 mmMặt trên: 3 x 120 mm / 2 x 140 mmMặt sau: 1 x 120 mmMặt bên: 2 x 120 mmVỏ bọc PSU: 2 x 120 mm,7,472.5 x 210 x 498 mm,MSI,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,...,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,https://cellphones.com.vn/tan-nhiet-khi-cpu-jonsbo-cr-1200e-rgb.html,Tản nhiệt khí CPU Jonsbo CR-1200E RGB,Jonsbo,Liên hệ,tan-nhiet-khi-cpu-jonsbo-cr-1200e-rgb,"Mua Tản nhiệt khí CPU Jonsbo CR-1200E RGB chính hãng - Giá rẻ, đảm bảo chất lượng, độ bền cao, hỗ trợ giao hàng tận nơi toàn quốc.",,,,,Jonsbo,Tản nhiệt khí,2000 RPM ±10%,≤ 26.5 dB(A),1 x 3-pin,,,,,,,,,,,,,,,,,,,,,,,,,,...,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2,https://cellphones.com.vn/mainboard-msi-b450m-a-pro-max-ii.html,Mainboard MSI B450M-A PRO MAX II,MSI,Liên hệ,MB.MSI.12,"Mua Mainboard MSI B450M-A PRO MAX II chính hãng - Giá rẻ, đảm bảo chất lượng, độ bền cao, hỗ trợ giao hàng tận nơi toàn quốc.",,,,M-ATX,MSI,,,,,AMD,AM4,2,DDR4,64GB,2x USB 2.0 (Phía sau) 4x USB 2.0 (Phía trước) 4x USB 5Gbps Type A (Phía sau) 2x USB 5Gbps Type A (Phía trước),Bộ điều khiển LAN Realtek 8125 2.5Gbps,Realtek ALC897 Codec Âm thanh độ nét cao 7.1 kênh,,,,,,,,,,,,,,,,,,...,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3,https://cellphones.com.vn/o-cung-ssd-transcend-110s-nvme-pcie-gen3-x4-512gb.html,Ổ cứng SSD Transcend 110S NVMe PCIe Gen3 x4 512GB,Transcend,Liên hệ,OC.TC.42,"Ổ cứng SSD Transcend 110S NVMe PCIe Gen3 x4 512GB chính hãng - Giá rẻ, đảm bảo chất lượng, độ bền cao, hỗ trợ giao hàng tận nơi toàn quốc.",,,,,Transcend,,,,,,,,,,,,,SSD,M2 PCIe,512GB,1700 MB/s,Microsoft Windows 7 trở lênLinux Kernel 2.6.31 trở lên,,,,,,,,,,,,,...,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
4,https://cellphones.com.vn/nguon-may-tinh-msi-mag-a850gl-pcie5-850w.html,Nguồn máy tính MSI Mag A850GL PCIe5 850W (80 Plus Gold),MSI,Liên hệ,nguon-may-tinh-msi-mag-a850gl-pcie5-850w,"Mua nguồn máy tính MSI Mag A850GL PCIe5 850W (80 Plus Gold) chính hãng - Giá rẻ, đảm bảo chất lượng, độ bền cao, hỗ trợ giao hàng tận nơi toàn quốc.",,,,,MSI,,,,,,,,,,,,,,,,,,850W,1 x Fan 12 cm,80 Plus Gold,,,,,,,,,,...,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,


## 3. Standardize Core Fields

In [5]:
def normalize_price(value: str):
    text = clean_text(value)
    if not text:
        return None
    lowered = text.lower()
    if lowered in {"liên hệ", "lien he", "đang cập nhật", "dang cap nhat"}:
        return None
    digits = re.sub(r"[^0-9]", "", text)
    return int(digits) if digits else None

required_cols = ["url", "ten_san_pham", "thuong_hieu", "gia_ban", "sku", "mo_ta_ngan", "category", "source_file"]
for col in required_cols:
    if col not in clean.columns:
        clean[col] = ""

before_rows = len(clean)
clean = clean[clean["ten_san_pham"].ne("")].copy()
clean["price_vnd"] = clean["gia_ban"].map(normalize_price)
clean["price_status"] = clean["price_vnd"].apply(lambda x: "available" if pd.notna(x) else "contact_or_missing")

clean = clean.drop_duplicates(subset=["url", "ten_san_pham", "sku"], keep="first").reset_index(drop=True)

print("Rows before:", before_rows)
print("Rows after removing empty product names and duplicates:", len(clean))

Rows before: 3037
Rows after removing empty product names and duplicates: 3014


## 4. Build Structured Specs

In [6]:
CORE_COLUMNS = {
    "product_id",
    "url",
    "ten_san_pham",
    "thuong_hieu",
    "gia_ban",
    "price_vnd",
    "price_status",
    "sku",
    "mo_ta_ngan",
    "category",
    "source_file",
}

if "product_id" in clean.columns:
    clean = clean.drop(columns=["product_id"])

clean.insert(0, "product_id", [f"P{idx + 1:06d}" for idx in range(len(clean))])

spec_columns = [col for col in clean.columns if col not in CORE_COLUMNS]

def row_specs(row) -> dict:
    specs = {}
    for col in spec_columns:
        value = row.get(col, "")
        if value:
            specs[col] = value
    return specs

def specs_to_text(specs: dict) -> str:
    return "; ".join(f"{key}: {value}" for key, value in specs.items())

clean["specs_json"] = clean.apply(lambda row: json.dumps(row_specs(row), ensure_ascii=False), axis=1)
clean["specs_text"] = clean["specs_json"].map(lambda value: specs_to_text(json.loads(value)))

products_clean = clean[[
    "product_id",
    "url",
    "ten_san_pham",
    "thuong_hieu",
    "gia_ban",
    "price_vnd",
    "price_status",
    "sku",
    "mo_ta_ngan",
    "category",
    "source_file",
    "specs_text",
    "specs_json",
]].copy()

products_clean.head()

,product_id,url,ten_san_pham,thuong_hieu,gia_ban,price_vnd,price_status,sku,mo_ta_ngan,category,source_file,specs_text,specs_json
0,P000001,https://cellphones.com.vn/case-may-tinh-msi-mag-forge-320r-airflow.html,Case máy tính MSI MAG FORGE 320R AIRFLOW,MSI,Liên hệ,NaN,contact_or_missing,CMT.MSI.02,"Mua Case máy tính MSI MAG FORGE 320R AIRFLOW chính hãng - Giá rẻ, đảm bảo chất lượng, độ bền cao, hỗ trợ giao hàng tận nơi toàn quốc.",phu_kien,thong_tin_links_accessory.csv,loại_case: Mid Tower; hỗ_trợ_quạt: Mặt trước: 3 x 120 mm / 2 x 140 mmMặt trên: 3 x 120 mm / 2 x 140 mmMặt sau: 1 x 120 mmMặt bên: 2 x 120 mmVỏ bọc PSU: 2 x ...,"{""loại_case"": ""Mid Tower"", ""hỗ_trợ_quạt"": ""Mặt trước: 3 x 120 mm / 2 x 140 mmMặt trên: 3 x 120 mm / 2 x 140 mmMặt sau: 1 x 120 mmMặt bên: 2 x 120 mmVỏ bọc P..."
1,P000002,https://cellphones.com.vn/tan-nhiet-khi-cpu-jonsbo-cr-1200e-rgb.html,Tản nhiệt khí CPU Jonsbo CR-1200E RGB,Jonsbo,Liên hệ,NaN,contact_or_missing,tan-nhiet-khi-cpu-jonsbo-cr-1200e-rgb,"Mua Tản nhiệt khí CPU Jonsbo CR-1200E RGB chính hãng - Giá rẻ, đảm bảo chất lượng, độ bền cao, hỗ trợ giao hàng tận nơi toàn quốc.",phu_kien,thong_tin_links_accessory.csv,hãng_sản_xuất: Jonsbo; loại_tản_nhiệt: Tản nhiệt khí; số_vòng_quay_của_quạt: 2000 RPM ±10%; độ_ồn_(dba): ≤ 26.5 dB(A); kết_nối: 1 x 3-pin,"{""hãng_sản_xuất"": ""Jonsbo"", ""loại_tản_nhiệt"": ""Tản nhiệt khí"", ""số_vòng_quay_của_quạt"": ""2000 RPM ±10%"", ""độ_ồn_(dba)"": ""≤ 26.5 dB(A)"", ""kết_nối"": ""1 x 3-pin""}"
2,P000003,https://cellphones.com.vn/mainboard-msi-b450m-a-pro-max-ii.html,Mainboard MSI B450M-A PRO MAX II,MSI,Liên hệ,NaN,contact_or_missing,MB.MSI.12,"Mua Mainboard MSI B450M-A PRO MAX II chính hãng - Giá rẻ, đảm bảo chất lượng, độ bền cao, hỗ trợ giao hàng tận nơi toàn quốc.",phu_kien,thong_tin_links_accessory.csv,kích_thước: M-ATX; hãng_sản_xuất: MSI; chipset: AMD; socket: AM4; khe_ram_tối_đa: 2; kiểu_ram_hỗ_trợ: DDR4; dung_lượng_ram_tối_đa: 64GB; số_cổng_usb: 2x USB...,"{""kích_thước"": ""M-ATX"", ""hãng_sản_xuất"": ""MSI"", ""chipset"": ""AMD"", ""socket"": ""AM4"", ""khe_ram_tối_đa"": ""2"", ""kiểu_ram_hỗ_trợ"": ""DDR4"", ""dung_lượng_ram_tối_đa""..."
3,P000004,https://cellphones.com.vn/o-cung-ssd-transcend-110s-nvme-pcie-gen3-x4-512gb.html,Ổ cứng SSD Transcend 110S NVMe PCIe Gen3 x4 512GB,Transcend,Liên hệ,NaN,contact_or_missing,OC.TC.42,"Ổ cứng SSD Transcend 110S NVMe PCIe Gen3 x4 512GB chính hãng - Giá rẻ, đảm bảo chất lượng, độ bền cao, hỗ trợ giao hàng tận nơi toàn quốc.",phu_kien,thong_tin_links_accessory.csv,hãng_sản_xuất: Transcend; loại_ổ_cứng: SSD; kết_nối_2: M2 PCIe; dung_lượng: 512GB; tốc_độ_đọc: 1700 MB/s; hỗ_trợ_hệ_điều_hành: Microsoft Windows 7 trở lênLi...,"{""hãng_sản_xuất"": ""Transcend"", ""loại_ổ_cứng"": ""SSD"", ""kết_nối_2"": ""M2 PCIe"", ""dung_lượng"": ""512GB"", ""tốc_độ_đọc"": ""1700 MB/s"", ""hỗ_trợ_hệ_điều_hành"": ""Micro..."
4,P000005,https://cellphones.com.vn/nguon-may-tinh-msi-mag-a850gl-pcie5-850w.html,Nguồn máy tính MSI Mag A850GL PCIe5 850W (80 Plus Gold),MSI,Liên hệ,NaN,contact_or_missing,nguon-may-tinh-msi-mag-a850gl-pcie5-850w,"Mua nguồn máy tính MSI Mag A850GL PCIe5 850W (80 Plus Gold) chính hãng - Giá rẻ, đảm bảo chất lượng, độ bền cao, hỗ trợ giao hàng tận nơi toàn quốc.",phu_kien,thong_tin_links_accessory.csv,hãng_sản_xuất: MSI; công_suất_nguồn: 850W; quạt_làm_mát: 1 x Fan 12 cm; tiêu_chuẩn_nguồn: 80 Plus Gold,"{""hãng_sản_xuất"": ""MSI"", ""công_suất_nguồn"": ""850W"", ""quạt_làm_mát"": ""1 x Fan 12 cm"", ""tiêu_chuẩn_nguồn"": ""80 Plus Gold""}"


## 5. Create Long-Format Specs Table

In [7]:
spec_rows = []
for _, row in products_clean.iterrows():
    specs = json.loads(row["specs_json"])
    for spec_name, spec_value in specs.items():
        spec_rows.append({
            "product_id": row["product_id"],
            "spec_name": spec_name,
            "spec_value": spec_value,
        })

specs_long = pd.DataFrame(spec_rows, columns=["product_id", "spec_name", "spec_value"])
print(specs_long.shape)
specs_long.head()

(30265, 3)


,product_id,spec_name,spec_value
0,P000001,loại_case,Mid Tower
1,P000001,hỗ_trợ_quạt,Mặt trước: 3 x 120 mm / 2 x 140 mmMặt trên: 3 x 120 mm / 2 x 140 mmMặt sau: 1 x 120 mmMặt bên: 2 x 120 mmVỏ bọc PSU: 2 x 120 mm
2,P000001,số_khe_cắm_pci,7
3,P000001,kích_thước,472.5 x 210 x 498 mm
4,P000001,hãng_sản_xuất,MSI


## 6. Example: Filter by Category and Spec

In [8]:
phone_ids = products_clean.loc[products_clean["category"].eq("dien_thoai"), "product_id"]

ram_8gb_ids = specs_long.loc[
    specs_long["product_id"].isin(phone_ids)
    & specs_long["spec_name"].eq("dung_lượng_ram")
    & specs_long["spec_value"].str.contains(r"\b8\s*GB\b", case=False, na=False),
    "product_id",
]

phone_ram_8gb = products_clean[products_clean["product_id"].isin(ram_8gb_ids)]
phone_ram_8gb[["product_id", "ten_san_pham", "thuong_hieu", "category", "specs_text", "url"]].head()

,product_id,ten_san_pham,thuong_hieu,category,specs_text,url
1476,P001477,OPPO K10 Pro,OPPO,dien_thoai,chipset: Qualcomm SM8350 Snapdragon 888 5G (5 nm); pin: Li-Po 5000 mAh (Không thể tháo rời); dung_lượng_ram: 8 GB; kích_thước_màn_hình: 6.62 inches; công_ng...,https://cellphones.com.vn/oppo-k10-pro.html
1477,P001478,Vivo iQOO Neo 8 Pro,Vivo,dien_thoai,chipset: Snapdragon 888+ 5G (5 nm); pin: 4500 mAh; dung_lượng_ram: 8 GB; kích_thước_màn_hình: 6.78 inches; công_nghệ_màn_hình: LTPO AMOLED; hệ_điều_hành: An...,https://cellphones.com.vn/vivo-iqoo-neo-8-pro.html
1479,P001480,Samsung Galaxy A07 8GB 256GB,Samsung,dien_thoai,chipset: MediaTek Helio G99; pin: Li-Ion 5000 mAh; dung_lượng_ram: 8 GB; kích_thước_màn_hình: 6.7 inches; công_nghệ_màn_hình: IPS LCD; hệ_điều_hành: Android...,https://cellphones.com.vn/dien-thoai-samsung-galaxy-a07-256gb.html
1481,P001482,Nubia V80 Max 8GB 128GB,Nubia,dien_thoai,chipset: Unisoc T7250; pin: 6000 mAh; dung_lượng_ram: 8 GB; kích_thước_màn_hình: 6.9 inches; công_nghệ_màn_hình: IPS LCD; camera_sau: 50 MP + 2 MP + 0.08 MP...,https://cellphones.com.vn/dien-thoai-nubia-v80-max-8gb-128gb.html
1482,P001483,Xperia 5 V,Hãng khác,dien_thoai,chipset: Qualcomm Snapdragon 8 Gen 2 (4 nm); pin: 5000 mAh; dung_lượng_ram: 8 GB; kích_thước_màn_hình: 6.1 inches; công_nghệ_màn_hình: OLED; hệ_điều_hành: A...,https://cellphones.com.vn/sony-xperia-5v.html


## 7. Export to processed/

In [9]:
products_clean.to_csv(OUT_DIR / "products_clean.csv", index=False, encoding="utf-8-sig")
products_clean.to_json(OUT_DIR / "products_clean.jsonl", orient="records", lines=True, force_ascii=False)

specs_long.to_csv(OUT_DIR / "product_specs_long.csv", index=False, encoding="utf-8-sig")
specs_long.to_json(OUT_DIR / "product_specs_long.jsonl", orient="records", lines=True, force_ascii=False)

category_counts = products_clean["category"].value_counts().rename_axis("category").reset_index(name="product_count")
category_counts.to_csv(OUT_DIR / "category_counts.csv", index=False, encoding="utf-8-sig")

report = {
    "input_files": [file.name for file in csv_files],
    "raw_rows": int(before_rows),
    "clean_rows": int(len(products_clean)),
    "removed_rows": int(before_rows - len(products_clean)),
    "spec_rows": int(len(specs_long)),
    "categories": category_counts.to_dict(orient="records"),
    "output_files": [
        "products_clean.csv",
        "products_clean.jsonl",
        "product_specs_long.csv",
        "product_specs_long.jsonl",
        "category_counts.csv",
        "processing_report.json",
    ],
}

with (OUT_DIR / "processing_report.json").open("w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("Saved cleaned data to:", OUT_DIR.resolve())
category_counts

Saved cleaned data to: /Users/huybui/Desktop/Smart Shopping Chatbot/processed


,category,product_count
0,dien_thoai,921
1,laptop,675
2,phu_kien,359
3,tai_nghe_bluetooth,301
4,man_hinh,291
5,may_tinh_bang,178
6,loa,151
7,pc,82
8,micro,56
